# 04. Machine Learning Model Training & Evaluation

**Football IQ - Player Performance Prediction & Scouting System**

Training, chronological validation, hyperparameter tuning, and metric comparison across Linear Regression, Random Forest, Gradient Boosting, and XGBoost models.


In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append("..")
from src.model_training import prepare_training_data, train_and_evaluate_models

# Load dataset
proc_path = os.path.join("..", "data", "processed", "players_processed.csv")
df = pd.read_csv(proc_path)

# Prepare chronological train/test split
data_dict = prepare_training_data(df)
print(f"Training Samples (Seasons 2021-24): {len(data_dict['X_train'])}")
print(f"Testing Samples (Season 2024-25): {len(data_dict['X_test'])}")
print(f"Number of Features: {len(data_dict['feature_cols'])}")


### 1. Train and Benchmark All Regressors


In [ ]:
results, best_model, best_model_name = train_and_evaluate_models(data_dict)

print(f"--- BEST MODEL: {best_model_name} ---\n")
metrics_table = []
for name, m in results.items():
    metrics_table.append({
        "Model": name,
        "MAE": m["MAE"],
        "MSE": m["MSE"],
        "RMSE": m["RMSE"],
        "R2": m["R2"]
    })
metrics_df = pd.DataFrame(metrics_table).sort_values("R2", ascending=False)
display(metrics_df)


### 2. Actual vs. Predicted Performance


In [ ]:
# Model predictions on test set
X_test = data_dict["X_test"]
y_test = data_dict["y_test"]
preds = best_model.predict(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, preds, alpha=0.5, color="teal")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
plt.xlabel("Actual Future Rating")
plt.ylabel("Predicted Future Rating")
plt.title(f"Actual vs. Predicted Future Rating ({best_model_name})")
plt.grid(True, alpha=0.3)
plt.show()


### 3. Feature Importance Analysis


In [ ]:
feat_imp = results[best_model_name]["top_features"]
imp_df = pd.DataFrame(feat_imp).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(imp_df["feature"], imp_df["importance"], color="steelblue")
plt.xlabel("Relative Importance (%)")
plt.title(f"Top 12 Predictive Features ({best_model_name})")
plt.tight_layout()
plt.show()
